# 🏛️ Government Schemes Dataset — Complete Analysis Notebook
### Scheme Recommendation | NLP | Machine Learning | Deep Learning
---

## 📦 Install & Import Dependencies

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly scikit-learn xgboost nltk wordcloud tensorflow transformers -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from wordcloud import WordCloud

# ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, precision_score,
                              recall_score, f1_score)
from xgboost import XGBClassifier

# DL
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('All libraries imported successfully.')
print(f'TensorFlow: {tf.__version__}')

---
## Section 1 — Data Collection Questions

### Q1. Load dataset and inspect available information

In [ ]:
# Load dataset — update path if running locally
DATA_PATH = 'Government_Schemes_Dataset.csv'  # or full path
df = pd.read_csv(DATA_PATH)

print('=== DATASET OVERVIEW ===')
print(f'Shape           : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage    : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print('\nColumn names:')
for col in df.columns:
    print(f'  {col}')
df.head(3)

### Q2 & Q7. Total schemes and numerical vs categorical features

In [ ]:
print(f'Total schemes in dataset: {len(df):,}')
print(f'\nNumerical columns  : {list(df.select_dtypes(include="number").columns)}')
print(f'Categorical columns: {list(df.select_dtypes(include="object").columns)}')

print('\nColumn descriptions:')
col_desc = {
    'scheme_name'   : 'Name of the government scheme',
    'slug'          : 'URL-friendly identifier',
    'details'       : 'Full textual description of the scheme',
    'benefits'      : 'What benefits the scheme provides',
    'eligibility'   : 'Who is eligible to apply',
    'application'   : 'How to apply (steps)',
    'documents'     : 'Required documents for application',
    'level'         : 'State or Central level scheme',
    'schemeCategory': 'Category of the scheme',
    'tags'          : 'Keywords/tags associated with scheme'
}
for col, desc in col_desc.items():
    print(f'  {col:<20}: {desc}')

### Q3 & Q8. Scheme categories and most frequent category

In [ ]:
cat_counts = df['schemeCategory'].value_counts()
print(f'Unique scheme categories: {df["schemeCategory"].nunique()}')
print('\nAll categories with counts:')
print(cat_counts.to_string())

print(f'\nMost frequent category: "{cat_counts.index[0]}" ({cat_counts.iloc[0]:,} schemes)')

### Q5 & Q6. State-level vs Central-level schemes

In [ ]:
level_counts = df['level'].value_counts()
print('Scheme levels:')
print(level_counts.to_string())

# National (Central) schemes are available nationwide
central = df[df['level'] == 'Central']
state   = df[df['level'] == 'State']
print(f'\nNationwide (Central) schemes: {len(central):,}')
print(f'State-specific schemes      : {len(state):,}')

# Q9: Useful columns for recommendation
print('\nColumns most useful for recommendation:')
useful = ['schemeCategory', 'level', 'eligibility', 'benefits', 'tags']
for col in useful:
    print(f'  ✓ {col}')

# Q10: Irrelevant columns
print('\nColumns to consider removing:')
print('  ✗ slug         — URL identifier, not informative for ML')
print('  ✗ Unnamed: 9   — entirely empty column')

---
## Section 2 — Data Preprocessing

### Q1–Q5. Missing values and duplicates

In [ ]:
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0].to_string())

print('\n=== DUPLICATES ===')
dups = df.duplicated().sum()
name_dups = df['scheme_name'].duplicated().sum()
print(f'Full duplicate rows   : {dups}')
print(f'Duplicate scheme names: {name_dups}')

# Visualize missing values
fig, ax = plt.subplots(figsize=(10, 4))
cols_with_missing = missing_df[missing_df['Missing Count'] > 0]
ax.barh(cols_with_missing.index, cols_with_missing['Missing %'], color='#e74c3c', alpha=0.8)
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Column', fontweight='bold')
for i, (idx, row) in enumerate(cols_with_missing.iterrows()):
    ax.text(row['Missing %'] + 0.1, i, f"{row['Missing Count']} ({row['Missing %']}%)", va='center')
plt.tight_layout()
plt.show()

### Q3 & Q6–Q12. Full preprocessing pipeline

In [ ]:
df_clean = df.copy()

# 1. Drop empty column
df_clean.drop(columns=['Unnamed: 9'], inplace=True, errors='ignore')
print('✓ Dropped empty column "Unnamed: 9"')

# 2. Handle missing values
df_clean['application'].fillna('Not specified', inplace=True)
df_clean['documents'].fillna('Not specified', inplace=True)
df_clean['tags'].fillna('', inplace=True)
print('✓ Filled missing text fields with "Not specified" / empty string')

# 3. Standardize level column
df_clean['level'] = df_clean['level'].str.strip().str.title()
print(f'✓ Standardized level column: {df_clean["level"].unique()}')

# 4. Standardize schemeCategory
df_clean['schemeCategory'] = df_clean['schemeCategory'].str.strip()
print('✓ Stripped whitespace from schemeCategory')

# 5. Remove full duplicates
before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print(f'✓ Removed {before - len(df_clean)} duplicate rows')

print(f'\nFinal clean dataset shape: {df_clean.shape}')

### Q8–Q9. Label encoding for categorical variables

In [ ]:
le_category = LabelEncoder()
le_level    = LabelEncoder()

df_clean['category_encoded'] = le_category.fit_transform(df_clean['schemeCategory'])
df_clean['level_encoded']    = le_level.fit_transform(df_clean['level'])

print('Label Encoding:')
print(f'  Level    : {dict(zip(le_level.classes_, le_level.transform(le_level.classes_)))}')
print(f'\n  schemeCategory → integer (first 10):')
mapping = dict(zip(le_category.classes_, le_category.transform(le_category.classes_)))
for k, v in list(mapping.items())[:10]:
    print(f'    {k[:50]:<52}: {v}')

### Q12. Text cleaning function

In [ ]:
stop_words  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

def clean_text(text):
    """Clean text: lowercase, remove special chars, stop words, lemmatize."""
    if pd.isna(text): return ''
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df_clean['clean_details']     = df_clean['details'].apply(clean_text)
df_clean['clean_eligibility'] = df_clean['eligibility'].apply(clean_text)
df_clean['clean_benefits']    = df_clean['benefits'].apply(clean_text)

print('Sample cleaned text:')
print('Original  :', df_clean['details'].iloc[0][:150])
print('\nCleaned   :', df_clean['clean_details'].iloc[0][:150])

---
## Section 3 — Exploratory Data Analysis (EDA)

### Q1 & Q2. Category frequency and state-level distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Top 15 categories
top_cats = df_clean['schemeCategory'].value_counts().head(15)
colors   = sns.color_palette('Set2', len(top_cats))
axes[0].barh(range(len(top_cats)), top_cats.values, color=colors)
axes[0].set_yticks(range(len(top_cats)))
axes[0].set_yticklabels([c[:45] for c in top_cats.index], fontsize=8)
axes[0].set_xlabel('Number of Schemes')
axes[0].set_title('Top 15 Scheme Categories', fontweight='bold')
for i, v in enumerate(top_cats.values):
    axes[0].text(v + 3, i, str(v), va='center', fontsize=8)

# State vs Central
level_data = df_clean['level'].value_counts()
axes[1].pie(level_data.values, labels=level_data.index, autopct='%1.1f%%',
            colors=['#3498db', '#e67e22'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('State vs Central Schemes', fontweight='bold')

plt.tight_layout()
plt.savefig('category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Q8–Q11. Agriculture-specific scheme exploration

In [ ]:
agri_df = df_clean[df_clean['schemeCategory'].str.contains('Agriculture', case=False, na=False)].copy()
print(f'Agriculture-related schemes: {len(agri_df)}')

# Keyword-based subcategorization
keyword_map = {
    'Crop Insurance'  : ['insurance', 'crop loss', 'fasal bima', 'compensation'],
    'Irrigation'      : ['irrigation', 'drip', 'sprinkler', 'water supply', 'canal'],
    'Organic Farming' : ['organic', 'natural farming', 'bio', 'compost', 'jaivik'],
    'Equipment'       : ['tractor', 'machinery', 'equipment', 'tools', 'mechanization'],
    'Women Farmers'   : ['women', 'female', 'mahila', 'self help group']
}

for sub_cat, keywords in keyword_map.items():
    pattern = '|'.join(keywords)
    count = df_clean['details'].str.lower().str.contains(pattern, na=False).sum()
    print(f'  {sub_cat:<20}: {count} schemes')

### Q17. Most impactful schemes (by benefit text length as proxy)

In [ ]:
df_clean['benefit_length'] = df_clean['benefits'].str.len()
df_clean['detail_length']  = df_clean['details'].str.len()

# Top 10 by benefit description richness
top_schemes = df_clean.nlargest(10, 'benefit_length')[['scheme_name', 'schemeCategory', 'level', 'benefit_length']]
print('Top 10 schemes with richest benefit descriptions:')
print(top_schemes[['scheme_name', 'schemeCategory', 'level']].to_string(index=False))

---
## Section 4 — Visualization

### Q1. Category distribution (Plotly interactive bar)

In [ ]:
cat_data = df_clean['schemeCategory'].value_counts().reset_index()
cat_data.columns = ['Category', 'Count']

fig = px.bar(cat_data.head(20), x='Count', y='Category', orientation='h',
             title='Scheme Category Distribution (Top 20)',
             color='Count', color_continuous_scale='Viridis',
             height=600)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

### Q2. State vs Central scheme distribution

In [ ]:
# Category breakdown by level
cat_level = df_clean.groupby(['schemeCategory', 'level']).size().unstack(fill_value=0)
cat_level['total'] = cat_level.sum(axis=1)
cat_level = cat_level.nlargest(15, 'total').drop(columns='total')

fig, ax = plt.subplots(figsize=(14, 7))
cat_level.plot(kind='barh', ax=ax, color=['#3498db', '#e67e22'], edgecolor='white', linewidth=0.3)
ax.set_title('Schemes by Category & Level (Top 15)', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Schemes')
ax.set_yticklabels([l[:45] for l in cat_level.index], fontsize=8)
ax.legend(title='Level', fontsize=10)
plt.tight_layout()
plt.show()

### Q4 & Q8. Heatmap — category vs level

In [ ]:
pivot = df_clean.pivot_table(index='schemeCategory', columns='level',
                              values='scheme_name', aggfunc='count', fill_value=0)
pivot = pivot[pivot.sum(axis=1) >= 10].sort_values('Central', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5,
            linecolor='white', ax=ax, cbar_kws={'label': 'Scheme count'})
ax.set_title('Heatmap: Scheme Count by Category × Level', fontsize=13, fontweight='bold')
ax.set_xlabel('Level')
ax.set_ylabel('Category')
ax.set_yticklabels([l[:45] for l in pivot.index], fontsize=8)
plt.tight_layout()
plt.savefig('category_level_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### Q9. Pie chart — category proportions (Plotly)

In [ ]:
top_n = 10
top_cats_pie = df_clean['schemeCategory'].value_counts().head(top_n)
others_count = df_clean['schemeCategory'].value_counts().iloc[top_n:].sum()
pie_labels = list(top_cats_pie.index) + ['Others']
pie_values = list(top_cats_pie.values) + [others_count]

fig = px.pie(names=pie_labels, values=pie_values,
             title=f'Scheme Category Proportions (Top {top_n} + Others)',
             hole=0.35)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

---
## Section 5 — Feature Engineering

### Q1–Q8. Engineered features for recommendation

In [ ]:
df_feat = df_clean.copy()

# 1. Level binary feature
df_feat['is_central'] = (df_feat['level'] == 'Central').astype(int)

# 2. Farmer type flags from eligibility text
def flag_keyword(series, keywords):
    pattern = '|'.join(keywords)
    return series.str.lower().str.contains(pattern, na=False).astype(int)

df_feat['targets_small_farmers']  = flag_keyword(df_feat['eligibility'], ['small farmer', 'marginal farmer', 'smallholder'])
df_feat['targets_women']          = flag_keyword(df_feat['eligibility'], ['women', 'female', 'mahila'])
df_feat['targets_sc_st']          = flag_keyword(df_feat['eligibility'], ['sc', 'st', 'scheduled caste', 'scheduled tribe', 'obc'])
df_feat['targets_fishermen']      = flag_keyword(df_feat['eligibility'], ['fishermen', 'fisherman', 'fisher'])

# 3. Scheme type flags from benefits/details
df_feat['has_financial_benefit']  = flag_keyword(df_feat['benefits'], ['subsidy', 'financial', 'grant', 'loan', 'assistance', '₹', 'rs.'])
df_feat['has_training_benefit']   = flag_keyword(df_feat['benefits'], ['training', 'skill', 'education', 'workshop'])
df_feat['has_insurance_benefit']  = flag_keyword(df_feat['benefits'], ['insurance', 'coverage', 'compensation'])

# 4. Text length features
df_feat['eligibility_len']  = df_feat['eligibility'].str.len().fillna(0)
df_feat['benefits_len']     = df_feat['benefits'].str.len().fillna(0)
df_feat['details_len']      = df_feat['details'].str.len().fillna(0)

# 5. Scheme complexity score (more docs = more complex application)
df_feat['doc_count'] = df_feat['documents'].apply(
    lambda x: len(str(x).split('.')) if pd.notna(x) else 0)

# 6. Popularity score — based on tag count
df_feat['tag_count'] = df_feat['tags'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) and x != '' else 0)

engineered = ['is_central', 'targets_small_farmers', 'targets_women', 'targets_sc_st',
              'has_financial_benefit', 'has_training_benefit', 'has_insurance_benefit',
              'eligibility_len', 'benefits_len', 'doc_count', 'tag_count']

print('Engineered features summary:')
print(df_feat[engineered].describe().round(2).to_string())

### Feature correlation heatmap

In [ ]:
numeric_features = ['is_central', 'targets_small_farmers', 'targets_women',
                    'has_financial_benefit', 'has_training_benefit',
                    'has_insurance_benefit', 'doc_count', 'tag_count']

corr = df_feat[numeric_features].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Feature Correlation Heatmap', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 6 — Machine Learning

### Prepare features and labels for classification

In [ ]:
# Use TF-IDF on combined text + engineered features
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
df_feat['combined_text'] = (df_feat['clean_eligibility'] + ' ' +
                             df_feat['clean_benefits'] + ' ' +
                             df_feat['clean_details'])

tfidf_matrix = tfidf.fit_transform(df_feat['combined_text'])
tfidf_df     = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

# Combine with engineered features
X_num = df_feat[numeric_features].reset_index(drop=True)
X     = pd.concat([tfidf_df.reset_index(drop=True), X_num], axis=1)

# Target: scheme category (top N classes only for balance)
top_cats_list = df_feat['schemeCategory'].value_counts().head(10).index.tolist()
df_ml = df_feat[df_feat['schemeCategory'].isin(top_cats_list)].reset_index(drop=True)
X_ml  = pd.concat([pd.DataFrame(tfidf.transform(df_ml['combined_text']).toarray(),
                                 columns=tfidf.get_feature_names_out()),
                   df_ml[numeric_features].reset_index(drop=True)], axis=1)

le_ml = LabelEncoder()
y_ml  = le_ml.fit_transform(df_ml['schemeCategory'])

X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_ml, test_size=0.2, random_state=42, stratify=y_ml)

print(f'Training samples : {len(X_train):,}')
print(f'Testing samples  : {len(X_test):,}')
print(f'Features         : {X_ml.shape[1]}')
print(f'Classes          : {len(le_ml.classes_)}')

### Q2–Q4. Train Decision Tree, Random Forest, XGBoost, KNN

In [ ]:
models = {
    'Decision Tree'  : DecisionTreeClassifier(max_depth=15, random_state=42),
    'Random Forest'  : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost'        : XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42,
                                      use_label_encoder=False, eval_metric='mlogloss'),
    'KNN'            : KNeighborsClassifier(n_neighbors=5)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall'   : recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1'       : f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'model'    : model,
        'y_pred'   : y_pred
    }
    print(f'{name:<18} | Acc: {results[name]["accuracy"]:.4f} | F1: {results[name]["f1"]:.4f}')

best_model_name = max(results, key=lambda k: results[k]['f1'])
print(f'\nBest model: {best_model_name} (F1={results[best_model_name]["f1"]:.4f})')

### Q8. Model comparison chart

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1']
model_names = list(results.keys())
x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(13, 6))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
for i, (metric, color) in enumerate(zip(metrics, colors)):
    values = [results[m][metric] for m in model_names]
    bars = ax.bar(x + i * width, values, width, label=metric.capitalize(), color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim([0, 1.1])
ax.set_ylabel('Score')
ax.set_title('ML Model Comparison — Scheme Classification', fontweight='bold', fontsize=13)
ax.legend()
ax.axhline(0.9, color='black', linestyle='--', alpha=0.3, label='0.90 target')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Q5. Feature importance (Random Forest)

In [ ]:
rf_model = results['Random Forest']['model']
importances = rf_model.feature_importances_
feat_names  = X_ml.columns.tolist()

feat_imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#e74c3c' if f in numeric_features else '#3498db' for f in feat_imp_df['feature']]
ax.barh(range(len(feat_imp_df)), feat_imp_df['importance'].values, color=colors, alpha=0.85)
ax.set_yticks(range(len(feat_imp_df)))
ax.set_yticklabels(feat_imp_df['feature'], fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 20 Feature Importances — Random Forest', fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#e74c3c', label='Engineered feature'),
    Patch(color='#3498db', label='TF-IDF feature')
])
plt.tight_layout()
plt.show()

---
## Section 7 — NLP Analysis

### Q3. Word cloud from scheme descriptions

In [ ]:
all_text = ' '.join(df_clean['clean_details'].dropna().tolist())

wc = WordCloud(width=1200, height=600, background_color='white',
               colormap='viridis', max_words=150, collocations=False)
wc.generate(all_text)

plt.figure(figsize=(16, 8))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in Scheme Descriptions', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('wordcloud_all.png', dpi=150, bbox_inches='tight')
plt.show()

### Q1–Q2. TF-IDF analysis and top terms per category

In [ ]:
# Per-category TF-IDF top terms
top_cats_tfidf = df_clean['schemeCategory'].value_counts().head(5).index.tolist()

fig, axes = plt.subplots(1, len(top_cats_tfidf), figsize=(20, 6))

for ax, cat in zip(axes, top_cats_tfidf):
    cat_text = df_clean[df_clean['schemeCategory'] == cat]['clean_details'].dropna()
    if len(cat_text) < 3:
        continue
    tfidf_cat = TfidfVectorizer(max_features=200, ngram_range=(1, 2))
    tfidf_cat.fit_transform(cat_text)
    terms = tfidf_cat.get_feature_names_out()
    scores = np.asarray(tfidf_cat.transform(cat_text).mean(axis=0)).flatten()
    top_idx = scores.argsort()[-12:][::-1]

    ax.barh(range(12), scores[top_idx], color='#2980b9', alpha=0.85)
    ax.set_yticks(range(12))
    ax.set_yticklabels([terms[i][:20] for i in top_idx], fontsize=8)
    ax.set_title(cat[:30], fontsize=8, fontweight='bold')
    ax.set_xlabel('Avg TF-IDF', fontsize=7)

plt.suptitle('Top TF-IDF Terms per Scheme Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Q7. Patterns in eligibility descriptions

In [ ]:
elig_text = ' '.join(df_clean['clean_eligibility'].dropna().tolist())
cv = CountVectorizer(max_features=30, ngram_range=(1, 2))
cv.fit([elig_text])
counts = cv.transform([elig_text]).toarray()[0]
top_elig_terms = pd.Series(counts, index=cv.get_feature_names_out()).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
top_elig_terms.plot(kind='bar', ax=ax, color='#27ae60', alpha=0.85, edgecolor='white')
ax.set_title('Most Common Terms in Eligibility Criteria', fontweight='bold', fontsize=13)
ax.set_xlabel('Term')
ax.set_ylabel('Frequency')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Section 8 — Deep Learning

### Q1–Q6. ANN / DNN for scheme classification

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
X_test_scaled  = scaler.transform(X_test.values)

NUM_CLASSES_ML = len(le_ml.classes_)

def build_dnn(input_dim, num_classes, hidden_layers=[512, 256, 128], dropout=0.4):
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for units in hidden_layers:
        x = layers.Dense(units, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='DNN_SchemeClassifier')

dnn = build_dnn(X_train_scaled.shape[1], NUM_CLASSES_ML)
dnn.compile(optimizer=keras.optimizers.Adam(1e-3),
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy'])
dnn.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

dnn_history = dnn.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

dnn_preds = np.argmax(dnn.predict(X_test_scaled), axis=1)
dnn_acc   = accuracy_score(y_test, dnn_preds)
dnn_f1    = f1_score(y_test, dnn_preds, average='weighted', zero_division=0)
print(f'\nDNN Test Accuracy: {dnn_acc:.4f}')
print(f'DNN Test F1 Score: {dnn_f1:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(dnn_history.history['accuracy']) + 1)

axes[0].plot(epochs, dnn_history.history['accuracy'],     label='Train', color='#3498db')
axes[0].plot(epochs, dnn_history.history['val_accuracy'], label='Val',   color='#e74c3c', linestyle='--')
axes[0].set_title('DNN Accuracy', fontweight='bold'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, dnn_history.history['loss'],         label='Train', color='#3498db')
axes[1].plot(epochs, dnn_history.history['val_loss'],     label='Val',   color='#e74c3c', linestyle='--')
axes[1].set_title('DNN Loss', fontweight='bold'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Deep Neural Network Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Sections 9 & 10 — Training Strategy & Evaluation

### Cross-validation and train/test split analysis

In [ ]:
print('=== CROSS-VALIDATION (5-fold) ===')
cv_models = {
    'Random Forest' : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost'       : XGBClassifier(n_estimators=100, random_state=42,
                                     use_label_encoder=False, eval_metric='mlogloss')
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X_ml, y_ml, cv=skf, scoring='f1_weighted', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<18} | F1: {scores.mean():.4f} ± {scores.std():.4f} | Folds: {scores.round(4)}')

# Plot CV results
fig, ax = plt.subplots(figsize=(10, 5))
for i, (name, scores) in enumerate(cv_results.items()):
    ax.plot(range(1, 6), scores, marker='o', label=name, linewidth=2)
    ax.axhline(scores.mean(), linestyle='--', alpha=0.4, color=f'C{i}')
ax.set_xticks(range(1, 6))
ax.set_xlabel('Fold')
ax.set_ylabel('F1 Score (weighted)')
ax.set_title('5-Fold Cross Validation Results', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Full evaluation — confusion matrix for best model

In [ ]:
best = results[best_model_name]
y_pred_best = best['y_pred']

print(f'=== {best_model_name} — Evaluation Report ===')
print(f'Accuracy  : {best["accuracy"]:.4f}')
print(f'Precision : {best["precision"]:.4f}')
print(f'Recall    : {best["recall"]:.4f}')
print(f'F1 Score  : {best["f1"]:.4f}')
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred_best,
                             target_names=le_ml.classes_,
                             zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[l[:20] for l in le_ml.classes_],
            yticklabels=[l[:20] for l in le_ml.classes_],
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('confusion_matrix_schemes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 11 — Scheme Recommendation System

### Q1–Q8. Farmer profile-based scheme recommendation

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Build TF-IDF matrix over all combined text
tfidf_rec = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
df_rec    = df_clean.copy()
df_rec['combined_text'] = (df_rec['clean_eligibility'] + ' ' +
                            df_rec['clean_benefits'] + ' ' +
                            df_rec['clean_details'])
tfidf_matrix_rec = tfidf_rec.fit_transform(df_rec['combined_text'])

def recommend_schemes(farmer_profile: dict, top_k: int = 5) -> pd.DataFrame:
    """
    Recommend government schemes based on a farmer profile dictionary.

    Parameters:
        farmer_profile: dict with keys like:
            - occupation     : e.g. 'farmer', 'fisherman'
            - category       : e.g. 'small farmer', 'marginal', 'women'
            - interest       : e.g. 'irrigation', 'crop insurance', 'equipment'
            - level          : 'Central', 'State', or 'Both'
            - scheme_category: e.g. 'Agriculture,Rural & Environment'
        top_k: number of recommendations to return

    Returns:
        DataFrame with top_k recommended schemes
    """
    # Build query text from profile
    query_parts = []
    for key in ['occupation', 'category', 'interest']:
        if key in farmer_profile:
            query_parts.append(farmer_profile[key])
    query_text = clean_text(' '.join(query_parts))

    query_vec = tfidf_rec.transform([query_text])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix_rec).flatten()

    df_scored = df_rec.copy()
    df_scored['similarity'] = sim_scores

    # Filter by level if specified
    if farmer_profile.get('level') in ['Central', 'State']:
        df_scored = df_scored[df_scored['level'] == farmer_profile['level']]

    # Filter by scheme category if specified
    if 'scheme_category' in farmer_profile:
        df_scored = df_scored[
            df_scored['schemeCategory'].str.contains(
                farmer_profile['scheme_category'], case=False, na=False)]

    top_schemes = df_scored.nlargest(top_k, 'similarity')[
        ['scheme_name', 'schemeCategory', 'level', 'benefits', 'similarity']
    ].reset_index(drop=True)

    return top_schemes

# ---- Demo: Farmer profile 1 ----
profile1 = {
    'occupation'     : 'farmer',
    'category'       : 'small marginal farmer',
    'interest'       : 'irrigation drip sprinkler subsidy',
    'level'          : 'Central',
    'scheme_category': 'Agriculture'
}
recs1 = recommend_schemes(profile1, top_k=5)
print('=== Recommended Schemes for Small Farmer (Irrigation) ===')
for i, row in recs1.iterrows():
    print(f'\n#{i+1}: {row["scheme_name"]}')
    print(f'   Category   : {row["schemeCategory"]}')
    print(f'   Level      : {row["level"]}')
    print(f'   Similarity : {row["similarity"]:.3f}')
    print(f'   Benefits   : {str(row["benefits"])[:120]}...')

In [ ]:
# ---- Demo: Farmer profile 2 (Women farmer) ----
profile2 = {
    'occupation' : 'women farmer',
    'category'   : 'self help group mahila',
    'interest'   : 'financial assistance training skill'
}
recs2 = recommend_schemes(profile2, top_k=5)
print('=== Recommended Schemes for Women Farmer ===')
for i, row in recs2.iterrows():
    print(f'#{i+1}: {row["scheme_name"][:70]} | Similarity: {row["similarity"]:.3f}')

# ---- Demo: Farmer profile 3 (Crop insurance) ----
profile3 = {
    'occupation' : 'farmer',
    'interest'   : 'crop insurance loss compensation natural disaster'
}
recs3 = recommend_schemes(profile3, top_k=5)
print('\n=== Recommended Schemes for Crop Insurance ===')
for i, row in recs3.iterrows():
    print(f'#{i+1}: {row["scheme_name"][:70]} | Similarity: {row["similarity"]:.3f}')

---
## Section 12 — Integration with Other Agricultural Modules

In [ ]:
def integrated_recommendation(farmer_data: dict) -> dict:
    """
    Simulated integration of:
      - Scheme recommendation
      - Crop recommendation (placeholder)
      - Disease detection (placeholder)
      - Soil/weather (placeholder)
    """
    output = {}

    # --- Module 1: Scheme Recommendation ---
    scheme_profile = {
        'occupation' : farmer_data.get('occupation', 'farmer'),
        'category'   : farmer_data.get('farmer_type', 'small farmer'),
        'interest'   : farmer_data.get('need', 'subsidy financial assistance')
    }
    output['recommended_schemes'] = recommend_schemes(scheme_profile, top_k=3)

    # --- Module 2: Crop Context (placeholder) ---
    crop = farmer_data.get('crop', 'rice')
    output['crop_info'] = f'Crop: {crop} — Check disease detection module for leaf analysis.'

    # --- Module 3: Weather (placeholder) ---
    output['weather_advisory'] = (
        f'For region {farmer_data.get("state", "Unknown")}, '
        f'integrate weather API to adjust eligibility for drought/flood schemes.'
    )

    # --- Module 4: Market price influence (placeholder) ---
    output['market_note'] = (
        f'If {crop} MSP is below market price, PM-KISAN or price support schemes apply.'
    )

    return output

# Test integration
farmer_input = {
    'occupation'  : 'farmer',
    'farmer_type' : 'small marginal farmer',
    'need'        : 'irrigation subsidy equipment loan',
    'crop'        : 'wheat',
    'state'       : 'Madhya Pradesh'
}

integrated_result = integrated_recommendation(farmer_input)
print('=== INTEGRATED RECOMMENDATION OUTPUT ===')
print('\n[Recommended Schemes]')
for i, row in integrated_result['recommended_schemes'].iterrows():
    print(f'  {i+1}. {row["scheme_name"][:60]}')
print(f'\n[Crop Info]     {integrated_result["crop_info"]}')
print(f'[Weather]       {integrated_result["weather_advisory"]}')
print(f'[Market]        {integrated_result["market_note"]}')

---
## Section 13 — Future Work

In [ ]:
future_work = [
    {'q':'Q1',  'area':'Real-time government updates',      'tech':'Government API / web scraping + scheduler',    'impact':'High'},
    {'q':'Q2',  'area':'AI chatbot for farmer awareness',   'tech':'LLM (GPT/Gemini) + Rasa/Dialogflow',          'impact':'High'},
    {'q':'Q3',  'area':'Multilingual support',              'tech':'IndicNLP / Google Translate API',              'impact':'High'},
    {'q':'Q4',  'area':'Voice assistants',                  'tech':'Speech-to-text + TTS + regional languages',    'impact':'High'},
    {'q':'Q5',  'area':'Smart agriculture platform',        'tech':'Microservices + REST API integration',         'impact':'Medium'},
    {'q':'Q6',  'area':'GIS mapping for subsidies',         'tech':'GeoPandas + Folium + state-wise data',         'impact':'Medium'},
    {'q':'Q7',  'area':'Generative AI guidance',            'tech':'RAG (Retrieval-Augmented Generation) + LLM',   'impact':'High'},
    {'q':'Q8',  'area':'Blockchain subsidy transparency',   'tech':'Ethereum / Hyperledger Fabric',                'impact':'Low'},
    {'q':'Q9',  'area':'Mobile application',                'tech':'Flutter + TFLite + offline capability',        'impact':'High'},
    {'q':'Q10', 'area':'AI policy impact prediction',       'tech':'Time-series forecasting + NLP policy analysis','impact':'Medium'},
]

df_fw = pd.DataFrame(future_work)
print(df_fw[['q','area','tech','impact']].to_string(index=False))

# Visualize
impact_colors = {'High': '#e74c3c', 'Medium': '#e67e22', 'Low': '#2ecc71'}
colors_fw = [impact_colors[i] for i in df_fw['impact']]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(df_fw['area'], [1]*len(df_fw), color=colors_fw, alpha=0.85, edgecolor='white')
ax.set_xlim([0, 1.4])
ax.set_xlabel('')
ax.set_title('Future Work Roadmap', fontweight='bold', fontsize=13)
for i, (bar, row) in enumerate(zip(bars, df_fw.itertuples())):
    ax.text(1.05, i, row.tech[:50], va='center', fontsize=8, color='#2c3e50')
ax.set_xticks([])
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=k) for k, c in impact_colors.items()],
          title='Impact', loc='lower right')
plt.tight_layout()
plt.savefig('future_work.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📊 Final Summary

In [ ]:
print('=' * 58)
print('   GOVERNMENT SCHEMES ANALYSIS — PROJECT SUMMARY')
print('=' * 58)

summary = {
    'Dataset'               : 'Government_Schemes_Dataset.csv',
    'Total schemes'         : f'{len(df_clean):,}',
    'Scheme categories'     : str(df_clean['schemeCategory'].nunique()),
    'Central schemes'       : str((df_clean['level'] == 'Central').sum()),
    'State schemes'         : str((df_clean['level'] == 'State').sum()),
    'ML models trained'     : 'Decision Tree, Random Forest, XGBoost, KNN',
    'Best ML model'         : f'{best_model_name} (F1={results[best_model_name]["f1"]:.3f})',
    'DNN accuracy'          : f'{dnn_acc:.4f}',
    'Recommendation system' : 'TF-IDF cosine similarity (profile-based)',
    'NLP features'          : 'TF-IDF, word cloud, per-category top terms',
    'Output files'          : 'category_distribution.png, model_comparison.png,\n'
                              '                  confusion_matrix_schemes.png,\n'
                              '                  wordcloud_all.png, dnn_training_curves.png'
}

for k, v in summary.items():
    print(f'  {k:<28}: {v}')

print('=' * 58)
print('All 13 sections addressed. Notebook complete.')